In [295]:
import pandas as pd

import asyncio
import httpx
from datetime import datetime, time, timedelta
import numpy as np

import sys
sys.path.append('..')
from sqlalchemy import create_engine, text
from sqlalchemy.dialects.postgresql import insert

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

In [296]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )
    
async def extraer_datos(endpoint, fecha_ini, fecha_fin):
    url = f"http://10.1.0.103:9080/Utea/{endpoint}"
    params = {
        "pStrFecIni": fecha_ini,
        "pStrFecFin": fecha_fin,
    }
    timeout = httpx.Timeout(10.0)
    now = datetime.now()
    formatted_now = now.strftime('%d/%m/%Y %H:%M:%S')
    try:
        async with httpx.AsyncClient(timeout=timeout) as client:
            response = await client.get(url, params=params)
            if response.status_code == 200:
                print("✅ " + formatted_now + f" {endpoint}: GET exitoso")
                return response.json()
            else:
                print("⚠️ " + formatted_now + f" {endpoint}: Error en obtener respuesta")
                return {}
    except httpx.RequestError as e:
        print("❌ " + formatted_now + f" {endpoint}: Error en conexion.")
        return {}


In [297]:
async def get_datos_balanza(fecha_inicio=None, fecha_fin=None):
    # Si no se pasan fechas, se calculan por defecto como las tenías antes
    if fecha_inicio is None:
        fecha_inicio = (datetime.now() - timedelta(days=3)).strftime('%Y-%m-%d')
    if fecha_fin is None:
        fecha_fin = (datetime.now() - timedelta(days=2)).strftime('%Y-%m-%d')
        
    # Extraer datos usando los parámetros de fecha_inicio (antes ayer) y fecha_fin (antes hoy)
    TrafCamBalanza = await extraer_datos("TrafCamBalanza", fecha_inicio, fecha_fin)
    
    df_trafcambalanza = pd.DataFrame(TrafCamBalanza)
    return df_trafcambalanza

def upsert_method(table, conn, keys, data_iter):
    """
    Función personalizada para permitir UPSERT en el to_sql de Pandas
    """
    # Convertir los datos a insertar en una lista de diccionarios
    data = [dict(zip(keys, row)) for row in data_iter]
    
    # Crear la sentencia INSERT de SQLAlchemy apuntando a tu tabla
    insert_stmt = insert(table.table).values(data)
    
    # Definir la lógica de actualización en caso de conflicto con la llave primaria
    upsert_stmt = insert_stmt.on_conflict_do_update(
        index_elements=['ticketId'], # La columna que es Llave Primaria
        set_={c.name: c for c in insert_stmt.excluded if c.name != 'ticketId'} # Actualiza el resto
    )
    
    # Ejecutar en la base de datos
    conn.execute(upsert_stmt)

In [298]:
PATH_OUTPUT_DATA = 'G:\Ingenio Azucarero Guabira S.A\COOR_GERENCIA_CANA - Parte_Horarios'

In [304]:
fecha_inicio="2026-05-31"
fecha_fin="2026-06-01"
balanza = await get_datos_balanza(fecha_inicio, fecha_fin)

✅ 01/06/2026 15:26:46 TrafCamBalanza: GET exitoso


In [305]:
# 1. Convertimos a numérico, forzando los errores a convertirse en NaN
valores_numericos = pd.to_numeric(balanza['ticketId'], errors='coerce')
# 2. Filtramos el DataFrame original: nos quedamos solo con los que NO son NaN
#    y aseguramos que sean enteros exactos (evitando decimales si los hubiera)
df_limpio = balanza[valores_numericos.notna() & (valores_numericos % 1 == 0)].copy()
# 3. Ahora que está limpio, convertimos la columna a tipo entero con seguridad
df_limpio['ticketId'] = df_limpio['ticketId'].astype(int)

In [306]:
df_limpio

,subsystId,ticketId,ticketItem,startDate,startTime,hora,startSapuser,netWeight,fullWeight,emptyWeight,...,nroingreso,fechaCosecha,datum,fechaQuemado,horaQuemado,programa,dateCupo,tipoCupo,idchofer,nombreChofer
0,001,26002518,0001,2026-05-31,0001-01-01T06:04:44,,USERTEC,31460.0,50180.0,18720.0,...,2699,2026-05-31,2026-05-31,0000-00-00,0001-01-01T00:00:00,,2026-05-30,Cupo Tolerancia,0005340746,Miguel Angel Romero
1,001,26002519,0001,2026-05-31,0001-01-01T06:06:32,,USERTEC,43650.0,65190.0,21540.0,...,2714,2026-05-31,2026-05-31,0000-00-00,0001-01-01T00:00:00,,2026-05-30,Cupo Tolerancia,0006335290,LORENZO COSSIO
2,001,26002520,0001,2026-05-31,0001-01-01T06:09:38,,USERTEC,55600.0,86390.0,30790.0,...,2565,2026-05-30,2026-05-31,0000-00-00,0001-01-01T00:00:00,,2026-05-30,Cupo Tolerancia,0006261046,Wilson Sandoval peña
3,001,26002521,0001,2026-05-31,0001-01-01T06:12:05,,USERTEC,56020.0,81490.0,25470.0,...,2518,2026-05-30,2026-05-31,0000-00-00,0001-01-01T00:00:00,,2026-05-30,Cupo Tolerancia,0001970161,JAVIER ANGEL PEDRAZA TORREZ
4,001,26002522,0001,2026-05-31,0001-01-01T06:13:38,,USERTEC,39660.0,58670.0,19010.0,...,2719,2026-05-31,2026-05-31,0000-00-00,0001-01-01T00:00:00,,2026-05-30,Cupo Tolerancia,0007674431,GILBERTO MEDINA GOMES
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
613,001,26003131,0001,2026-06-01,0001-01-01T15:20:55,,USERTEC,0.0,69040.0,0.0,...,3295,2026-06-01,2026-06-01,0000-00-00,0001-01-01T00:00:00,,2026-06-01,Cupo Tolerancia,0008894962,RICHARD VILAJA
614,001,26003132,0001,2026-06-01,0001-01-01T15:22:19,,USERTEC,0.0,63500.0,0.0,...,3305,2026-06-01,2026-06-01,0000-00-00,0001-01-01T00:00:00,,2026-06-01,Cupo Tolerancia,0008920315,JOSE LUIS SAUCEDO NEGRETE
615,001,26003133,0001,2026-06-01,0001-01-01T15:23:33,,USERTEC,0.0,59880.0,0.0,...,3273,2026-06-01,2026-06-01,0000-00-00,0001-01-01T00:00:00,,2026-06-01,Cupo Tolerancia,0008199254,Ricardo Escarzo
616,001,26003134,0001,2026-06-01,0001-01-01T15:25:14,,USERTEC,0.0,54870.0,0.0,...,3281,2026-06-01,2026-06-01,0000-00-00,0001-01-01T00:00:00,,2026-06-01,Cupo Tolerancia,0004685015,JOCOB SALVATIERRA


In [307]:
engine = obtener_engine()
# Así es como aplicas tu función
df_limpio.to_sql(
    name='trafcambalanza', 
    con=engine, 
    if_exists='append', # 'append' es obligatorio para que use el método personalizado
    schema='datos_iag',
    index=False, 
    method=upsert_method # <-- Aquí llamas a tu función
)